In [1]:
using JuMP, HiGHS, LinearAlgebra

In [2]:
m = Model(HiGHS.Optimizer)
@variable(m, x₁ ≥ 0)
@variable(m, x₂ ≥ 0)
@objective(m, Max, 2x₁+4x₂)
@constraint(m, c1, 4x₁+3x₂ ≤ 120)
@constraint(m, c2, x₁+2x₂ ≤ 40)
@constraint(m, c3, x₂ ≤ 16)
print(m)


Max 2 x₁ + 4 x₂
Subject to
 c1 : 4 x₁ + 3 x₂ ≤ 120
 c2 : x₁ + 2 x₂ ≤ 40
 c3 : x₂ ≤ 16
 x₁ ≥ 0
 x₂ ≥ 0


In [3]:
set_silent(m)
optimize!(m)
is_solved_and_feasible(m)

true

In [4]:
value(x₁), value(x₂), objective_value(m)

(8.0, 16.0, 80.0)

In [5]:
dual_status(m)

FEASIBLE_POINT::ResultStatusCode = 1

In [6]:
dual(c1), dual(c2), dual(c3)

(0.0, -2.0, -0.0)

In [7]:
shadow_price(c1),shadow_price(c2), shadow_price(c3)

(-0.0, 2.0, 0.0)

In [8]:
dual.(all_constraints(m, include_variable_in_set_constraints = true))

5-element Vector{Float64}:
  0.0
 -2.0
 -0.0
  0.0
  0.0

In [9]:
all_variables(m)

2-element Vector{VariableRef}:
 x₁
 x₂

In [10]:
reduced_cost(x₁)

-0.0

In [11]:
map(var -> name(var) => reduced_cost(var), all_variables(m))

2-element Vector{Pair{String, Float64}}:
 "x₁" => -0.0
 "x₂" => -0.0

In [12]:
map(var->name(var) => shadow_price(var), all_constraints(m, include_variable_in_set_constraints = false))

3-element Vector{Pair{String, Float64}}:
 "c1" => -0.0
 "c2" => 2.0
 "c3" => 0.0

In [13]:
[ xi => get_attribute(xi, MOI.VariableBasisStatus()) for xi in all_variables(m) ]

2-element Vector{Pair{VariableRef, MathOptInterface.BasisStatusCode}}:
 x₁ => MathOptInterface.BASIC
 x₂ => MathOptInterface.BASIC

In [14]:
MOI.get(m, MOI.ConstraintBasisStatus(), c1), MOI.get(m, MOI.ConstraintBasisStatus(), c2),  MOI.get(m, MOI.ConstraintBasisStatus(), c3)

(MathOptInterface.BASIC, MathOptInterface.NONBASIC, MathOptInterface.NONBASIC)

In [15]:
map(c-> name(c) => MOI.get(m, MOI.ConstraintBasisStatus(), c), all_constraints(m, include_variable_in_set_constraints = false))

3-element Vector{Pair{String, MathOptInterface.BasisStatusCode}}:
 "c1" => MathOptInterface.BASIC
 "c2" => MathOptInterface.NONBASIC
 "c3" => MathOptInterface.NONBASIC

In [16]:
reduced_cost(x₁), reduced_cost(x₂)

(-0.0, -0.0)

In [25]:
m3 = Model(HiGHS.Optimizer)
@variable(m3, x₁ ≥ 0)
@variable(m3, x₂ ≥ 0)
@variable(m3, x₃ ≥ 0)
@variable(m3, x₄ ≥ 0)
@objective(m3, Max, 2x₁ + 4x₂ + 3x₃ + 4x₄)
@constraint(m3, c1, x₁ + 2x₂ + x₃ + 2x₄ ≤ 10)
@constraint(m3, c2, 3x₁ + x₂ ≤ 9)
@constraint(m3, c3, x₃ + x₄ ≤ 4)
print(m3)

Max 2 x₁ + 4 x₂ + 3 x₃ + 4 x₄
Subject to
 c1 : x₁ + 2 x₂ + x₃ + 2 x₄ ≤ 10
 c2 : 3 x₁ + x₂ ≤ 9
 c3 : x₃ + x₄ ≤ 4
 x₁ ≥ 0
 x₂ ≥ 0
 x₃ ≥ 0
 x₄ ≥ 0


In [26]:
set_silent(m3)
optimize!(m3)
is_solved_and_feasible(m3)

true

In [28]:
value(x₁), value(x₂), value(x₃), value(x₄), objective_value(m3)

(0.0, 3.0, 4.0, 0.0, 24.0)

## $n$-queens

In [29]:
m2 = Model(HiGHS.Optimizer)
n = 4
P = [ i+j == n+1 ? 1 : 0 for i=1:n, j=1:n]
@variable(m2, x[1:n,1:n], Bin)
@constraint(m2, c1, mapslices(sum, x, dims = [1]) .<= ones(1,n))
@constraint(m2, c2, mapslices(sum, x, dims = [2]) .<= ones(n,1))
@constraint(m2, c3, [sum(diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@constraint(m2, c4, [sum(P*diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@objective(m2, Max, sum(x[1:n,1:n]))
# @objective(m2, Max, sum([rand(0.99:0.001:1.01)*x[i,j] for i=1:n, j=1:n]))
print(m2)

Max x[1,1] + x[2,1] + x[3,1] + x[4,1] + x[1,2] + x[2,2] + x[3,2] + x[4,2] + x[1,3] + x[2,3] + x[3,3] + x[4,3] + x[1,4] + x[2,4] + x[3,4] + x[4,4]
Subject to
 c1 : x[1,1] + x[2,1] + x[3,1] + x[4,1] ≤ 1
 c1 : x[1,2] + x[2,2] + x[3,2] + x[4,2] ≤ 1
 c1 : x[1,3] + x[2,3] + x[3,3] + x[4,3] ≤ 1
 c1 : x[1,4] + x[2,4] + x[3,4] + x[4,4] ≤ 1
 c2 : x[1,1] + x[1,2] + x[1,3] + x[1,4] ≤ 1
 c2 : x[2,1] + x[2,2] + x[2,3] + x[2,4] ≤ 1
 c2 : x[3,1] + x[3,2] + x[3,3] + x[3,4] ≤ 1
 c2 : x[4,1] + x[4,2] + x[4,3] + x[4,4] ≤ 1
 c3 : x[3,1] + x[4,2] ≤ 1
 c3 : x[2,1] + x[3,2] + x[4,3] ≤ 1
 c3 : x[1,1] + x[2,2] + x[3,3] + x[4,4] ≤ 1
 c3 : x[1,2] + x[2,3] + x[3,4] ≤ 1
 c3 : x[1,3] + x[2,4] ≤ 1
 c4 : x[2,1] + x[1,2] ≤ 1
 c4 : x[3,1] + x[2,2] + x[1,3] ≤ 1
 c4 : x[4,1] + x[3,2] + x[2,3] + x[1,4] ≤ 1
 c4 : x[4,2] + x[3,3] + x[2,4] ≤ 1
 c4 : x[4,3] + x[3,4] ≤ 1
 x[1,1] binary
 x[2,1] binary
 x[3,1] binary
 x[4,1] binary
 x[1,2] binary
 x[2,2] binary
 x[3,2] binary
 x[4,2] binary
 x[1,3] binary
 x[2,3] binary
 x[3,3] b

In [30]:
set_silent(m2)
optimize!(m2)
is_solved_and_feasible(m2)

true

In [31]:
round.(Int, value.(x))

4×4 Matrix{Int64}:
 0  0  1  0
 1  0  0  0
 0  0  0  1
 0  1  0  0

In [34]:
map(var -> name(var) => reduced_cost(var), all_variables(m2))

ErrorException: Unable to query reduced cost of variable because model does not have duals available.

In [32]:
Dict(
    xi => get_attribute(xi, MOI.VariableBasisStatus()) for
    xi in all_variables(m2)
)

Dict{VariableRef, MathOptInterface.BasisStatusCode} with 16 entries:
  x[4,2] => NONBASIC
  x[2,3] => NONBASIC
  x[2,1] => NONBASIC
  x[4,3] => NONBASIC
  x[3,1] => NONBASIC
  x[3,4] => NONBASIC
  x[4,1] => NONBASIC
  x[1,4] => NONBASIC
  x[2,4] => NONBASIC
  x[4,4] => NONBASIC
  x[1,3] => NONBASIC
  x[1,2] => NONBASIC
  x[3,3] => NONBASIC
  x[2,2] => NONBASIC
  x[1,1] => NONBASIC
  x[3,2] => NONBASIC

In [33]:
map(c-> name(c) => MOI.get(m2, MOI.ConstraintBasisStatus(), c), all_constraints(m2, include_variable_in_set_constraints = false))

18-element Vector{Pair{String, MathOptInterface.BasisStatusCode}}:
 "c1" => MathOptInterface.BASIC
 "c1" => MathOptInterface.BASIC
 "c1" => MathOptInterface.BASIC
 "c1" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c2" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c3" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC
 "c4" => MathOptInterface.BASIC

In [36]:
dual.(c1)

1×4 Matrix{Float64}:
 0.0  0.0  0.0  0.0

In [39]:
termination_status(m2)

OPTIMAL::TerminationStatusCode = 1

In [37]:
reduced_cost.(x)

ErrorException: Unable to query reduced cost of variable because model does not have duals available.